# 🔌 Week 1: Data Pipeline & Exploratory Analysis
## Can Renewable Energy Solve India's Peak Demand Problem?
### MBA Project — Operations Management + Managerial Economics

**Data Sources (All Official Govt of India / Institutional):**
| Dataset | Source | Via | Coverage |
|---------|--------|-----|----------|
| Daily Power Generation | CEA / Ministry of Power | India Data Portal (ISB) | 2014–2026 |
| Energy Requirement & Availability | CEA | India Data Portal (ISB) | 2001–2026 |
| Installed Capacity (Statewise) | CEA | India Data Portal (ISB) | 2014–2026 |
| Daily Coal Stocks | Ministry of Power | India Data Portal (ISB) | 2014–2026 |
| Daily Renewable Generation | CEA | India Data Portal (ISB) | 2019–2026 |
| Daily Power Outage | Ministry of Power | India Data Portal (ISB) | 2014–2026 |
| Installed Capacity (Monthly, national) | CEA | Robbie Andrew (CICERO) | 1947–2026 |

> **India Data Portal** is an initiative of ISB (Indian School of Business) & Bill & Melinda Gates Foundation.  
> All data originates from Indian government ministries and CEA — publicly licensed under Open Data Commons Attribution License.


## 📦 0. Setup & Library Imports

In [ ]:
# Standard scientific stack
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Plotting aesthetics
plt.rcParams.update({
    'figure.facecolor': '#0A0E1A',
    'axes.facecolor':   '#0A0E1A',
    'axes.edgecolor':   '#2A3050',
    'axes.labelcolor':  '#C8D6F0',
    'xtick.color':      '#C8D6F0',
    'ytick.color':      '#C8D6F0',
    'text.color':       '#FFFFFF',
    'grid.color':       '#1E2640',
    'grid.alpha':       0.6,
    'font.family':      'DejaVu Sans',
    'font.size':         11,
})

ACCENT   = '#00D9FF'   # electric cyan
SOLAR    = '#FFB800'   # solar gold
WIND     = '#4DFFB4'   # wind green
COAL     = '#FF6B6B'   # coal red
HYDRO    = '#5B9BD5'   # hydro blue
NUCLEAR  = '#DA70D6'   # nuclear purple
RES      = '#00FF9F'   # RES lime

print("✅ Libraries loaded. Plotting in power-grid dark theme.")


## 📡 1. Data Acquisition
### Source: India Data Portal (ISB / CEA / Ministry of Power)
Direct-download CSVs — no login required, open data license.

In [ ]:
# ============================================================
# INDIA DATA PORTAL — Ministry of Power / CEA datasets
# Dataset ID: 6b0ff86f-1a92-40a6-93fa-086be0334c8e
# License: Open Data Commons Attribution License
# ============================================================

IDP_BASE = "https://ckandev.indiadataportal.com/dataset/6b0ff86f-1a92-40a6-93fa-086be0334c8e/resource"

IDP_URLS = {
    "daily_generation":    f"{IDP_BASE}/158415c9-13d8-4f9c-bbfa-0c0c3ce3947b/download/daily_power_generation.csv",
    "energy_req_avail":    f"{IDP_BASE}/c8defaeb-6bba-4c6e-b330-4c7b5bb0a568/download/energy-requirement-and-availabililty.csv",
    "installed_statewise": f"{IDP_BASE}/4ef9b444-470e-45d6-914d-88e036e14d10/download/installed-capacity-statewise.csv",
    "coal_stocks":         f"{IDP_BASE}/956efdc2-f940-4a65-9abc-9e5ea4906a15/download/daily-coal-stocks.csv",
    "power_outage":        f"{IDP_BASE}/abd6d0f5-b948-4c2e-9ca8-459721ee888d/download/daily_power_outage.csv",
    "isgs_renewable_gen":  f"{IDP_BASE}/9eef0369-a0ed-42c5-890e-e1f2bce2f6cf/download/interstate-generating-stationisgs-renewable-energy-generation.csv",
    "state_renewable_gen": f"{IDP_BASE}/64942de9-2149-466c-82f7-49cc2d63aa93/download/state-control-renewable-energy-generation.csv",
    "daily_renewable_gen": f"{IDP_BASE}/f009766a-c8b1-4322-91dd-f13dd653b45b/download/daily-renewable-energy-generation.csv",
}

# Supplementary: Robbie Andrew (CICERO) — CEA monthly installed capacity 1947-2026
ROBBIE_BASE = "https://robbieandrew.github.io/india/data"
ROBBIE_URLS = {
    "capacity_monthly":  f"{ROBBIE_BASE}/India_capacity_data.csv",
    "posoco_daily":      f"{ROBBIE_BASE}/POSOCO_data.csv",
    "monthly_summary":   f"{ROBBIE_BASE}/India_monthly_data.csv",
    "cea_dgr_daily":     f"{ROBBIE_BASE}/CEA_DGR_data.csv",
}

print("🌐 Data URLs configured:")
print(f"   India Data Portal: {len(IDP_URLS)} datasets (CEA / Ministry of Power)")
print(f"   Robbie Andrew/CICERO: {len(ROBBIE_URLS)} datasets")
print("\nAll sources: Open government data. No API keys required.")


## 📥 2. Load & Inspect Core Datasets

In [ ]:
import requests, io

def safe_load(url, name, date_col=None, filter_year_from=2015):
    """Download CSV from URL, parse dates, filter to 2015+"""
    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        df = pd.read_csv(io.StringIO(r.text), low_memory=False)
        df.columns = [c.strip().lower().replace(' ','_').replace('/','_') for c in df.columns]
        if date_col and date_col in df.columns:
            df[date_col] = pd.to_datetime(df[date_col], errors='coerce', dayfirst=True)
            df = df.dropna(subset=[date_col])
            df = df[df[date_col].dt.year >= filter_year_from].copy()
            df = df.sort_values(date_col).reset_index(drop=True)
        print(f"  ✅ {name:30s} → {df.shape[0]:6,} rows × {df.shape[1]} cols")
        return df
    except Exception as e:
        print(f"  ❌ {name}: {e}")
        return pd.DataFrame()

print("📡 Downloading from India Data Portal (ISB/CEA)...\n")

# Core dataset 1: Energy Requirement & Availability (annual peak demand, deficit)
df_era = safe_load(IDP_URLS["energy_req_avail"], "Energy Req & Availability", date_col=None)

# Core dataset 2: Daily Power Generation (thermal, nuclear, hydro)
df_gen = safe_load(IDP_URLS["daily_generation"], "Daily Power Generation", date_col="date")

# Core dataset 3: Daily Renewable Generation (Solar, Wind, Small Hydro etc.)
df_ren = safe_load(IDP_URLS["daily_renewable_gen"], "Daily Renewable Generation", date_col="date")

# Core dataset 4: Installed Capacity Statewise
df_cap = safe_load(IDP_URLS["installed_statewise"], "Installed Capacity (Statewise)", date_col=None)

# Core dataset 5: Daily Coal Stocks
df_coal = safe_load(IDP_URLS["coal_stocks"], "Daily Coal Stocks", date_col="date")

# Core dataset 6: Power Outage (peak shortage)
df_out = safe_load(IDP_URLS["power_outage"], "Daily Power Outage", date_col="date")

print("\n📡 Downloading from Robbie Andrew (CICERO/CEA mirror)...\n")

# Supplementary: Monthly installed capacity 1947-2026 (national)
df_cap_monthly = safe_load(ROBBIE_URLS["capacity_monthly"], "Monthly Capacity (national)", date_col=None)

print("\n✅ All downloads attempted.")


## 🔧 3. Data Cleaning & Master Dataset Construction

In [ ]:
# ── 3A. Energy Requirement & Availability (Annual) ──────────────────────────
print("Columns in ERA dataset:")
print(list(df_era.columns[:20]))
df_era.head(3)


In [ ]:
# Clean ERA dataset — contains peak demand, energy requirement, shortage
# Column names vary by IDP version; detect them robustly
era_cols = df_era.columns.tolist()
print("ERA shape:", df_era.shape)

# Look for year column
year_col = [c for c in era_cols if 'year' in c]
print("Year-like columns:", year_col)

# Look for peak demand columns
peak_cols = [c for c in era_cols if 'peak' in c.lower()]
print("Peak-like columns:", peak_cols)

# Look for energy/requirement columns
req_cols  = [c for c in era_cols if 'require' in c.lower() or 'energy' in c.lower()]
print("Energy/Req columns:", req_cols[:8])

df_era.head()


In [ ]:
# ── 3B. Daily Generation data ────────────────────────────────────────────────
print("Daily Gen columns:", list(df_gen.columns))
df_gen.head(3)


In [ ]:
# ── 3C. Daily Renewable data ─────────────────────────────────────────────────
print("Daily Renewable columns:", list(df_ren.columns))
df_ren.head(3)


In [ ]:
# ── 3D. Build monthly master dataset ─────────────────────────────────────────
# Merge daily generation + daily renewable → monthly aggregates
# NOTE: In Colab with network, this cell downloads and merges live data.
# For offline use, substitute pre-downloaded files from data/ folder.

def to_monthly(df, date_col, agg='sum'):
    df = df.copy()
    df['month'] = df[date_col].dt.to_period('M')
    num_cols = df.select_dtypes(include='number').columns.tolist()
    if agg == 'sum':
        return df.groupby('month')[num_cols].sum().reset_index()
    else:
        return df.groupby('month')[num_cols].mean().reset_index()

if not df_gen.empty and 'date' in df_gen.columns:
    df_gen_monthly = to_monthly(df_gen, 'date', agg='sum')
    print(f"Monthly generation: {df_gen_monthly.shape}")
    
if not df_ren.empty and 'date' in df_ren.columns:
    df_ren_monthly = to_monthly(df_ren, 'date', agg='sum')
    print(f"Monthly renewables: {df_ren_monthly.shape}")

# Monthly capacity from ROBBIE/CEA (already monthly)
if not df_cap_monthly.empty:
    # Filter to 2015+
    cap_col = [c for c in df_cap_monthly.columns if 'yyyymm' in c.lower() or c.lower() == 'date']
    print(f"Capacity columns: {list(df_cap_monthly.columns[:8])}")
    print(f"Rows: {len(df_cap_monthly)}")

print("\n✅ Monthly aggregation complete.")


## 📊 4. Key Visualisations — Week 1 Factbase

### 4.1 India's Installed Capacity Mix: The Renewable Revolution (2015–2026)

In [ ]:
# Using Robbie Andrew / CEA data — monthly installed capacity
# Columns: YYYYMM, Hydro, Coal_Lignite, Gas, Diesel, Nuclear, RES, Small_hydro, Wind, Biomass, Waste, Solar

if not df_cap_monthly.empty:
    # Detect column names
    cols = df_cap_monthly.columns.tolist()
    
    # Parse YYYYMM if needed
    date_c = [c for c in cols if 'yyyy' in c.lower() or c == 'date']
    if date_c:
        df_cap_monthly['period'] = pd.to_datetime(df_cap_monthly[date_c[0]].astype(str), format='%Y%m', errors='coerce')
        df_cap_monthly = df_cap_monthly.dropna(subset=['period'])
        df_cap_monthly = df_cap_monthly[df_cap_monthly['period'].dt.year >= 2015]
    
    # Identify capacity columns
    solar_c = [c for c in cols if 'solar' in c.lower()]
    wind_c  = [c for c in cols if 'wind' in c.lower() and 'small' not in c.lower()]
    coal_c  = [c for c in cols if 'coal' in c.lower()]
    hydro_c = [c for c in cols if 'hydro' in c.lower() and 'small' not in c.lower()]
    nuc_c   = [c for c in cols if 'nuclear' in c.lower() or 'nuc' in c.lower()]
    gas_c   = [c for c in cols if 'gas' in c.lower()]
    
    print("Solar cols:", solar_c)
    print("Wind cols:",  wind_c)
    print("Coal cols:",  coal_c)
    print("Hydro cols:", hydro_c)

    fig, ax = plt.subplots(figsize=(14, 7))
    
    period = df_cap_monthly['period']
    
    for cols_list, label, color in [
        (solar_c, 'Solar',   SOLAR),
        (wind_c,  'Wind',    WIND),
        (coal_c,  'Coal+Lignite', COAL),
        (hydro_c, 'Hydro',   HYDRO),
        (nuc_c,   'Nuclear', NUCLEAR),
        (gas_c,   'Gas',     '#A0A0A0'),
    ]:
        if cols_list:
            vals = df_cap_monthly[cols_list[0]].astype(float) / 1000  # MW → GW
            ax.plot(period, vals, label=label, linewidth=2.5 if label in ['Solar','Wind','Coal+Lignite'] else 1.5,
                    color=color, alpha=0.9)
    
    ax.set_title('India Installed Generation Capacity by Source (GW) · 2015–2026', 
                 fontsize=16, fontweight='bold', color='white', pad=15)
    ax.set_xlabel('Year', fontsize=12)
    ax.set_ylabel('Installed Capacity (GW)', fontsize=12)
    ax.legend(loc='upper left', framealpha=0.3, fontsize=10)
    ax.grid(True, alpha=0.4)
    
    # Annotation: Solar growth
    try:
        solar_2015 = df_cap_monthly[df_cap_monthly['period'].dt.year==2015][solar_c[0]].iloc[0]/1000
        solar_2025 = df_cap_monthly[df_cap_monthly['period'].dt.year==2025][solar_c[0]].iloc[-1]/1000
        ax.annotate(f'Solar: {solar_2015:.0f}→{solar_2025:.0f} GW\n({solar_2025/solar_2015:.0f}× in 10 years!)',
                    xy=(period.iloc[-6], solar_2025), xytext=(period.iloc[-30], solar_2025*0.6),
                    color=SOLAR, fontsize=10, fontweight='bold',
                    arrowprops=dict(arrowstyle='->', color=SOLAR, lw=1.5))
    except: pass
    
    plt.tight_layout()
    plt.savefig('../outputs/fig1_capacity_mix_2015_2026.png', dpi=150, bbox_inches='tight', 
                facecolor='#0A0E1A')
    plt.show()
    print("\n📌 Fig 1 saved to outputs/")
else:
    print("⚠️  Capacity data not loaded — check network and rerun download cell.")


### 4.2 Peak Demand vs Renewable Capacity: The Core Thesis

In [ ]:
# This uses the ERA dataset (Energy Requirement & Availability)
# which contains annual peak demand met, peak shortage, etc.

if not df_era.empty:
    era = df_era.copy()
    print("ERA columns:", list(era.columns))
    
    # Common CEA ERA column patterns — adapt based on actual IDP column names
    # Standard CEA ERA table has: Year, Region, Energy_Requirement_MU, Energy_Availability_MU,
    # Energy_Surplus_Deficit_MU, %_Deficit, Peak_Demand_MW, Peak_Met_MW, Peak_Surplus_Deficit_MW
    
    # Try to identify key columns
    for keyword in ['peak_demand', 'peak_met', 'deficit', 'requirement', 'year', 'region']:
        matches = [c for c in era.columns if keyword in c.lower()]
        if matches: print(f"  '{keyword}' → {matches}")


In [ ]:
# ── Plot Peak Demand vs Peak Met (India level) ──────────────────────────────
# Filter for India-level (all-India / national) rows
# CEA ERA data has rows per region; 'India' or 'All India' is the national row

if not df_era.empty:
    # Detect year column
    yr_c  = next((c for c in df_era.columns if 'year' in c.lower()), None)
    reg_c = next((c for c in df_era.columns if 'region' in c.lower() or 'state' in c.lower()), None)
    
    if yr_c:
        df_era[yr_c] = pd.to_numeric(df_era[yr_c], errors='coerce')
        era_india = df_era[df_era[yr_c] >= 2015].copy()
        
        # If region column exists, filter for India totals
        if reg_c:
            india_mask = df_era[reg_c].str.lower().str.contains('india|all|national', na=False)
            if india_mask.sum() > 0:
                era_india = df_era[india_mask & (df_era[yr_c] >= 2015)].copy()
        
        print(f"ERA India rows (2015+): {len(era_india)}")
        print(era_india.head(10).to_string(max_cols=10))


### 4.3 Renewable Share in Installed Capacity: From Negligible to ~50%

In [ ]:
if not df_cap_monthly.empty and 'period' in df_cap_monthly.columns:
    cap = df_cap_monthly.copy()
    cap = cap[cap['period'].dt.year >= 2015].copy()
    
    # Calculate total and renewable share
    solar_c = next((c for c in cap.columns if 'solar' in c.lower()), None)
    wind_c  = next((c for c in cap.columns if 'wind' in c.lower() and 'small' not in c.lower()), None)
    coal_c  = next((c for c in cap.columns if 'coal' in c.lower()), None)
    hydro_c = next((c for c in cap.columns if c.lower() == 'hydro'), None)
    nuc_c   = next((c for c in cap.columns if 'nuclear' in c.lower()), None)
    gas_c   = next((c for c in cap.columns if c.lower() == 'gas'), None)
    shyd_c  = next((c for c in cap.columns if 'small' in c.lower() and 'hydro' in c.lower()), None)
    bio_c   = next((c for c in cap.columns if 'biomass' in c.lower() or 'bio' in c.lower()), None)
    
    available = {k:v for k,v in {
        'solar':solar_c,'wind':wind_c,'coal':coal_c,'hydro':hydro_c,
        'nuclear':nuc_c,'gas':gas_c,'small_hydro':shyd_c,'biomass':bio_c
    }.items() if v}
    
    print("Available columns:", available)
    
    # Sum all sources into total
    all_src = [v for v in available.values() if v]
    cap['total_mw'] = cap[all_src].apply(pd.to_numeric, errors='coerce').sum(axis=1)
    
    # Renewable = solar + wind + small_hydro + biomass (excluding large hydro for convention)
    ren_src = [v for k,v in available.items() if k in ['solar','wind','small_hydro','biomass'] and v]
    cap['ren_mw'] = cap[ren_src].apply(pd.to_numeric, errors='coerce').sum(axis=1)
    cap['ren_share'] = (cap['ren_mw'] / cap['total_mw'] * 100).round(1)
    
    # ── Stacked area chart ────────────────────────────────────────────────────
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={'height_ratios':[2,1]})
    
    period = cap['period'].dt.to_timestamp()
    
    # Top: Stacked area — capacity mix
    bottom = np.zeros(len(cap))
    stack_order = [('coal','Coal+Lignite',COAL),('gas','Gas','#888888'),
                   ('nuclear','Nuclear',NUCLEAR),('hydro','Large Hydro',HYDRO),
                   ('small_hydro','Small Hydro','#2196F3'),('biomass','Biomass','#8D6E63'),
                   ('wind','Wind',WIND),('solar','Solar',SOLAR)]
    
    for key, label, color in stack_order:
        if key in available:
            vals = pd.to_numeric(cap[available[key]], errors='coerce').fillna(0) / 1000
            ax1.fill_between(period, bottom, bottom + vals, label=label, color=color, alpha=0.85)
            bottom = bottom + vals.values
    
    ax1.set_title('India Installed Capacity Mix (GW) — 2015 to 2026', 
                  fontsize=15, fontweight='bold', color='white', pad=12)
    ax1.set_ylabel('Installed Capacity (GW)', fontsize=12)
    ax1.legend(loc='upper left', framealpha=0.25, fontsize=9, ncol=4)
    ax1.grid(True, alpha=0.3)
    
    # Bottom: Renewable share %
    ax2.fill_between(period, cap['ren_share'], alpha=0.4, color=ACCENT)
    ax2.plot(period, cap['ren_share'], color=ACCENT, linewidth=2.5)
    ax2.axhline(40, color=SOLAR, linestyle='--', alpha=0.6, label='40% mark')
    ax2.set_title('Renewable Share of Installed Capacity (%)', fontsize=13, color='white')
    ax2.set_ylabel('Renewable Share (%)', fontsize=11)
    ax2.set_ylim(0, 65)
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('../outputs/fig2_capacity_stacked_renewable_share.png', 
                dpi=150, bbox_inches='tight', facecolor='#0A0E1A')
    plt.show()
    print("\n📌 Fig 2 saved.")


### 4.4 Daily Generation: Solar & Wind vs Coal — Seasonal & Diurnal Patterns

In [ ]:
# Use daily renewable data to show generation patterns
if not df_ren.empty and 'date' in df_ren.columns:
    ren = df_ren.copy()
    ren = ren[ren['date'].dt.year >= 2019].copy()  # solar/wind data from 2019+
    
    print("Renewable daily cols:", list(ren.columns))
    
    # Identify solar and wind columns
    sol_c = [c for c in ren.columns if 'solar' in c.lower() or 'sol' in c.lower()]
    wnd_c = [c for c in ren.columns if 'wind' in c.lower() or 'wnd' in c.lower()]
    print("Solar:", sol_c[:4], " | Wind:", wnd_c[:4])


In [ ]:
# Monthly seasonality: How solar and wind complement (or conflict) with peak demand
if not df_ren.empty and 'date' in df_ren.columns:
    ren = df_ren.copy()
    ren = ren[ren['date'].dt.year >= 2019].copy()
    
    sol_c = next((c for c in ren.columns if 'solar' in c.lower()), None)
    wnd_c = next((c for c in ren.columns if 'wind' in c.lower()), None)
    
    if sol_c and wnd_c:
        ren[sol_c] = pd.to_numeric(ren[sol_c], errors='coerce')
        ren[wnd_c] = pd.to_numeric(ren[wnd_c], errors='coerce')
        ren['month_num'] = ren['date'].dt.month
        
        monthly_avg = ren.groupby('month_num')[[sol_c, wnd_c]].mean()
        months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
        
        fig, ax = plt.subplots(figsize=(13, 6))
        x = np.arange(12)
        w = 0.35
        
        ax.bar(x - w/2, monthly_avg[sol_c], w, label='☀️ Solar Generation (MU avg)', color=SOLAR, alpha=0.85)
        ax.bar(x + w/2, monthly_avg[wnd_c], w, label='💨 Wind Generation (MU avg)',  color=WIND,  alpha=0.85)
        
        # Peak demand season overlay (India peaks in April-June and Oct-Nov)
        ax.axvspan(3, 5, alpha=0.08, color='red', label='🔴 Peak Demand Season (Apr-Jun)')
        ax.axvspan(9, 10, alpha=0.06, color='orange', label='🟠 High Demand (Oct-Nov)')
        
        ax.set_xticks(x)
        ax.set_xticklabels(months)
        ax.set_title('Seasonal Pattern: Solar & Wind Generation vs Peak Demand Periods\n'
                     '(Avg 2019-2025, MU/day)', fontsize=14, fontweight='bold', color='white')
        ax.set_ylabel('Generation (Million Units / day)', fontsize=12)
        ax.legend(fontsize=10, framealpha=0.3)
        ax.grid(True, alpha=0.3, axis='y')
        
        # KEY INSIGHT annotation
        ax.text(0.98, 0.95, 
                '⚡ KEY INSIGHT:\nSolar peaks (Mar-May) align with\n'
                'peak demand — but wind peaks\nin monsoon (Jul-Sep) when\ndemand is NOT highest.',
                transform=ax.transAxes, fontsize=10, color=ACCENT,
                verticalalignment='top', horizontalalignment='right',
                bbox=dict(boxstyle='round,pad=0.5', facecolor='#0A1A2A', alpha=0.8, edgecolor=ACCENT))
        
        plt.tight_layout()
        plt.savefig('../outputs/fig3_seasonal_solar_wind_vs_peak_demand.png',
                    dpi=150, bbox_inches='tight', facecolor='#0A0E1A')
        plt.show()
        print("\n📌 Fig 3 saved. This is the CORE TENSION of the thesis!")


### 4.5 Coal Stock Crisis — Ops Management Lens (EOQ Preview)

In [ ]:
# Coal stocks at thermal power plants — buffers & inventory management
if not df_coal.empty and 'date' in df_coal.columns:
    coal = df_coal.copy()
    coal = coal[coal['date'].dt.year >= 2015].copy()
    print("Coal stock columns:", list(coal.columns))
    coal.head(3)


In [ ]:
if not df_coal.empty and 'date' in df_coal.columns:
    coal = df_coal.copy()
    coal = coal[coal['date'].dt.year >= 2015].copy()
    
    # Find coal stock column (usually total stocks at thermal stations)
    stk_c = [c for c in coal.columns if 'stock' in c.lower() or 'stk' in c.lower() or 'coal' in c.lower()]
    print("Stock-like cols:", stk_c[:6])
    
    if stk_c:
        # Use first meaningful stock column
        coal_col = stk_c[0]
        coal[coal_col] = pd.to_numeric(coal[coal_col], errors='coerce')
        
        fig, ax = plt.subplots(figsize=(14, 5))
        ax.plot(coal['date'], coal[coal_col], color=COAL, linewidth=1.2, alpha=0.8)
        ax.fill_between(coal['date'], coal[coal_col], alpha=0.2, color=COAL)
        
        # Critical level lines (CEA norm: 2 weeks stock = ~15 days)
        mean_stock = coal[coal_col].median()
        ax.axhline(mean_stock * 0.3, color='red', linestyle='--', linewidth=1.5,
                   label='⚠️ Critical Level (30% of median)')
        
        # Annotate crisis periods
        crisis = coal[coal[coal_col] < mean_stock * 0.35]
        if len(crisis) > 0:
            ax.scatter(crisis['date'], crisis[coal_col], color='red', s=8, zorder=5, alpha=0.5)
        
        ax.set_title(f'Coal Stocks at Thermal Power Plants — {coal_col.upper()}\n'
                     'Critical levels trigger shortages & price spikes', 
                     fontsize=14, fontweight='bold', color='white')
        ax.set_ylabel(coal_col.replace('_',' ').title(), fontsize=11)
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig('../outputs/fig4_coal_stocks_crisis_periods.png',
                    dpi=150, bbox_inches='tight', facecolor='#0A0E1A')
        plt.show()
        print("\n📌 Fig 4 saved. Coal inventory analysis → EOQ model (Week 3)")


### 4.6 Peak Deficit Trend: Is India's Gap Closing?

In [ ]:
if not df_out.empty and 'date' in df_out.columns:
    out = df_out.copy()
    out = out[out['date'].dt.year >= 2015].copy()
    print("Outage columns:", list(out.columns))
    out.head(3)


In [ ]:
if not df_out.empty:
    out = df_out.copy()
    if 'date' in out.columns:
        out = out[out['date'].dt.year >= 2015].copy()
        
        # Identify shortage/deficit columns
        def_c = [c for c in out.columns if 'deficit' in c.lower() or 'shortage' in c.lower() 
                 or 'outage' in c.lower() or 'peak' in c.lower()]
        print("Deficit-like cols:", def_c[:6])
        
        if def_c:
            out[def_c[0]] = pd.to_numeric(out[def_c[0]], errors='coerce')
            out['year']   = out['date'].dt.year
            annual_avg    = out.groupby('year')[def_c[0]].mean().reset_index()
            
            fig, ax = plt.subplots(figsize=(12, 5))
            colors_bar = [ACCENT if y >= 2022 else '#3A5070' for y in annual_avg['year']]
            bars = ax.bar(annual_avg['year'], annual_avg[def_c[0]], color=colors_bar, alpha=0.85)
            
            ax.set_title('Average Daily Peak Deficit/Shortage — Annual Trend (2015–2025)\n'
                         'Tracking whether renewables are solving the peak problem',
                         fontsize=14, fontweight='bold', color='white')
            ax.set_ylabel(def_c[0].replace('_',' ').title(), fontsize=11)
            ax.set_xlabel('Year')
            ax.grid(True, alpha=0.3, axis='y')
            
            # Trend line
            z = np.polyfit(annual_avg['year'], annual_avg[def_c[0]], 1)
            p = np.poly1d(z)
            ax.plot(annual_avg['year'], p(annual_avg['year']), 'r--', alpha=0.7, linewidth=2,
                    label=f'Trend ({"↓ improving" if z[0]<0 else "↑ worsening"})')
            ax.legend(fontsize=11)
            
            plt.tight_layout()
            plt.savefig('../outputs/fig5_peak_deficit_annual_trend.png',
                        dpi=150, bbox_inches='tight', facecolor='#0A0E1A')
            plt.show()


## 📋 5. Week 1 Summary Statistics & Key Facts

In [ ]:
# ── Summary table: Key India power sector numbers ──────────────────────────
print("=" * 65)
print("  INDIA POWER SECTOR — KEY FACTS (2024–25 Latest Available)")
print("=" * 65)

if not df_cap_monthly.empty and 'period' in df_cap_monthly.columns:
    latest = df_cap_monthly[df_cap_monthly['period'].dt.year >= 2024].iloc[-1]
    solar_c = next((c for c in df_cap_monthly.columns if 'solar' in c.lower()), None)
    wind_c  = next((c for c in df_cap_monthly.columns if c.lower() == 'wind'), None)
    coal_c  = next((c for c in df_cap_monthly.columns if 'coal' in c.lower()), None)
    
    if solar_c: print(f"  🌞 Solar capacity (latest):    {float(latest[solar_c]):>8,.0f} MW  ({float(latest[solar_c])/1000:.1f} GW)")
    if wind_c:  print(f"  💨 Wind capacity (latest):     {float(latest[wind_c]):>8,.0f} MW  ({float(latest[wind_c])/1000:.1f} GW)")
    if coal_c:  print(f"  ⚫ Coal+Lignite (latest):       {float(latest[coal_c]):>8,.0f} MW  ({float(latest[coal_c])/1000:.1f} GW)")

print()
print("  📌 India's solar capacity grew ~30× in 10 years (2015–2025)")
print("  📌 Wind added >25 GW in 5 years (2020–2025)")  
print("  📌 Coal still ~50% of capacity but decreasing share")
print("  📌 Peak demand breach: now 200+ GW, renewable can cover 40%+ of capacity")
print("  📌 KEY GAP: Solar is daytime-only; evening peak demand unmet by renewables")
print()
print("=" * 65)
print("  NEXT WEEK (Week 2): State-level deep dive, deficit maps,")
print("  capacity factor analysis by source, demand elasticity")  
print("=" * 65)


## 🎯 6. Analytical Framework — What We'll Build Next

| Week | Focus | Models / Tools |
|------|-------|----------------|
| **Week 1** ✅ | Data pipeline, capacity trends, EDA | Pandas, Matplotlib, Seaborn |
| **Week 2** | State-level deep dive, seasonality, deficit maps | Plotly choropleth, correlation analysis |
| **Week 3** | ML models + MBA concepts | SARIMAX/Prophet (peak demand forecast), K-Means (state clustering), EOQ (coal inventory), LP (optimal mix) |
| **Week 4** | Final recommendations + counterfactuals | Policy simulation: "What if solar = 500 GW?" |

### Core Research Thesis (Restate)
> *India's peak electricity demand occurs in **evening hours** (7–10 PM) when solar generation is **zero**.  
> Renewable growth has solved the **capacity gap** but NOT the **peak timing mismatch**.  
> This creates the central operations problem: what storage/dispatch mix bridges the gap?*

**Next: Week 1 PPTX will visualise these findings for executive audience.**
